# Praktikum 1: Klasifikasi Biner Risiko Burnout dengan CNN 1D pada Dataset PBL

## Langkah 1 - Import Library

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

## Langkah 2 - Pra Pengolahan Data

In [2]:
df = pd.read_csv('burnout_dataset1.csv')
df_sample, _ = train_test_split(
    df, train_size=10000, stratify=df['Burnout_Risk'], random_state=42
)

num_cols = df_sample.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in num_cols if c not in ['Person_ID', 'Burnout_Score']]

X_raw = SimpleImputer(strategy='median').fit_transform(df_sample[feature_cols])
X_scaled = StandardScaler().fit_transform(X_raw)

# Reshape menjadi (samples, n_features, 1) sebagai input Conv1D
X = X_scaled.reshape(X_scaled.shape[0], X_scaled.shape[1], 1)

# Target biner: 1 = High, 0 = Non-High (Low / Moderate)
y = (df_sample['Burnout_Risk'] == 'High').astype(np.float32).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}, Proporsi High: {y.mean():.4f}")

X_train: (8000, 19, 1), X_test: (2000, 19, 1), Proporsi High: 0.1970


## Langkah 3 - Pembuatan Model CNN 1D

In [3]:
tf.keras.utils.set_random_seed(42)
cnn = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1], 1)),
    tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu'),
    tf.keras.layers.MaxPool1D(pool_size=2, strides=2),
    tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu'),
    tf.keras.layers.MaxPool1D(pool_size=2, strides=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=128, activation='relu'),
    tf.keras.layers.Dense(units=1, activation='sigmoid'),
])

cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 19, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 9, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 9, 32)          │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 4, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,873 (77.63 KB)

 Trainable params: 19,873 (77.63 KB)

 Non-trainable params: 0 (0.00 B)

## Langkah 4 - Fit Model CNN

In [4]:
history = cnn.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=64,
    verbose=1,
)
test_loss, test_acc = cnn.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

Epoch 1/15


  1/125 ━━━━━━━━━━━━━━━━━━━━ 2:41 1s/step - accuracy: 0.2969 - loss: 0.7289

 15/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7531 - loss: 0.5524 

 30/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7865 - loss: 0.4775

 45/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8170 - loss: 0.4239

 61/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8430 - loss: 0.3660

... (170 outputs left out)

103/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9331 - loss: 0.1419

117/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9343 - loss: 0.1391

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9342 - loss: 0.1386 - val_accuracy: 0.9060 - val_loss: 0.2083


Test Loss: 0.2083, Test Accuracy: 0.9060


## Langkah 5 - Prediksi pada 1 Sampel Data

In [5]:
sample_idx = 0
single_x = np.expand_dims(X_test[sample_idx], axis=0)
result = cnn.predict(single_x, verbose=0)
pred_label = 'High' if result[0][0] >= 0.5 else 'Non-High'
actual_label = 'High' if y_test[sample_idx] == 1.0 else 'Non-High'
print(f"Probabilitas High: {result[0][0]:.4f} | Prediksi: {pred_label} | Aktual: {actual_label}")

Probabilitas High: 0.0003 | Prediksi: Non-High | Aktual: Non-High
